# 🧪 Sesión 37 · Pruebas Automatizadas E2E con Postman / Newman CLI

**Desarrollo de Aplicaciones Web (DAB-2201)** · TecNM Cancún  
**Docente:** MCC. Iván Márquez Larios (`ijmarquezl`)  
**Repositorio:** https://github.com/ijmarquezl/tecnm-desarrollo-web

> **La pregunta de hoy:** ¿Cómo demuestras que todo el flujo funciona de principio a fin?

---

## 🖥️ Así se verá el resultado final

```text
$ newman run coleccion.json -e entorno.json
→ registro        [201 Created]  ✅ pasa
→ login           [200 OK]       ✅ pasa
→ crear libro     [201 Created]  ✅ pasa
→ listar libros   [200 OK]       ✅ pasa
→ perfil          [200 OK]       ✅ pasa

5 requests, 0 failures
La variable {{token}} se guardó en el login y se usó en perfil.
```

_Una prueba E2E no solo mira el código HTTP: también valida el contenido del JSON._

**No hay pasos intermedios.** El cuaderno declara el resultado que debe existir al final; tú decides la estructura, los modelos y el orden de tu implementación. La celda de verificación compara tu resultado con este contrato.

---

## 🎯 Al finalizar podrás

- Construir una colección Postman con variables.
- Encadenar registro, login y acceso autenticado.
- Automatizar la suite con Newman.

## ✅ Criterios de aceptación

El verificador comprobará estos criterios sobre los archivos que produzcas:

- [ ] La colección tiene aserciones y variables
- [ ] El entorno define la URL base
- [ ] La colección tiene al menos cinco peticiones

## 🧩 El reto

Crea una colección Postman/Newman que recorra registro, login, creación, listado y perfil usando el token guardado.

**Requisitos del contrato**

- La colección usa el esquema de Postman v2.1.
- Contiene al menos cinco peticiones en orden.
- El login guarda el token en una variable de entorno.
- Las peticiones protegidas usan {{token}}.
- Cada petición incluye una aserción sobre el código HTTP y el JSON.

> **Pista conceptual:** En Postman, el orden de los elementos es el orden de ejecución. Guarda el token con `pm.environment.set` y úsalo con `{{token}}` en la siguiente petición.

## 📦 Entregable

Cuaderno en verde y bitácora `auditorias/auditoria_sesion37.md` con la salida de Newman.


## ⚙️ Preparación del entorno

Ejecuta esta celda una sola vez. Instala las herramientas del track **Python / FastAPI** y descarga el motor de verificación del repositorio. No necesitas instalar nada en tu computadora: todo corre en Colab.


In [1]:
import os, sys
print("⏳ Preparando el entorno de Desarrollo Web...")
!pip install -q fastapi uvicorn httpx pydantic "passlib[bcrypt]" bcrypt python-jose[cryptography] PyJWT "sqlalchemy[asyncio]" greenlet aiosqlite alembic python-multipart
!wget -q -O verificar.py https://raw.githubusercontent.com/ijmarquezl/tecnm-desarrollo-web/c275993ba69ef2a8cf650b2f127b55c8356279d7/laboratorios/verificar.py
if not os.path.exists("verificar.py") or os.path.getsize("verificar.py") < 1000:
    raise RuntimeError("No se pudo descargar verificar.py. Revisa tu conexión o abre el cuaderno desde el repositorio del curso.")
print("✅ Entorno listo")
print("Python:", sys.version.split()[0])
print("Motor de verificación:", os.path.getsize("verificar.py"), "bytes")


⏳ Preparando el entorno de Desarrollo Web...
✅ Entorno listo
Python: 3.13.18
Motor de verificación: 13395 bytes


## 🧩 El reto: escribe el resultado

Crea el o los archivos del contrato. No hay una plantilla de pasos: el esqueleto solo te da el punto de partida. Todo lo que sigue lo decides tú.

**Archivos que debes producir:** `coleccion.json`, `entorno.json`


### 📄 `coleccion.json`

Colección de pruebas.

In [2]:
%%writefile coleccion.json
{
  "info": {
    "name": "Biblioteca E2E",
    "schema": "https://schema.getpostman.com/json/collection/v2.1.0/collection.json"
  },
  "item": [
    {
      "name": "1. Login",
      "event": [{
        "listen": "test",
        "script": {
          "type": "text/javascript",
          "exec": [
            "pm.test('Login responde 200', function () {",
            "  pm.response.to.have.status(200);",
            "});",
            "const json = pm.response.json();",
            "pm.environment.set('token', json.token);"
          ]
        }
      }],
      "request": {
        "method": "POST",
        "header": [{"key": "Content-Type", "value": "application/json"}],
        "body": {
          "mode": "raw",
          "raw": "{\"email\": \"admin@biblioteca.test\", \"password\": \"secreto123\"}"
        },
        "url": {"raw": "{{base_url}}/auth/login", "host": ["{{base_url}}"], "path": ["auth", "login"]}
      }
    },
    {
      "name": "2. Crear libro",
      "event": [{
        "listen": "test",
        "script": {
          "type": "text/javascript",
          "exec": [
            "pm.test('Libro creado (201)', function () {",
            "  pm.response.to.have.status(201);",
            "});",
            "pm.environment.set('libro_id', pm.response.json().id);"
          ]
        }
      }],
      "request": {
        "method": "POST",
        "header": [
          {"key": "Content-Type", "value": "application/json"},
          {"key": "Authorization", "value": "Bearer {{token}}"}
        ],
        "body": {
          "mode": "raw",
          "raw": "{\"titulo\": \"Cien años de soledad\", \"autor\": \"Gabriel García Márquez\"}"
        },
        "url": {"raw": "{{base_url}}/libros", "host": ["{{base_url}}"], "path": ["libros"]}
      }
    },
    {
      "name": "3. Listar libros",
      "event": [{
        "listen": "test",
        "script": {
          "type": "text/javascript",
          "exec": [
            "pm.test('Lista responde 200', function () {",
            "  pm.response.to.have.status(200);",
            "});",
            "pm.test('La respuesta es un arreglo', function () {",
            "  pm.expect(pm.response.json()).to.be.an('array');",
            "});"
          ]
        }
      }],
      "request": {
        "method": "GET",
        "header": [{"key": "Authorization", "value": "Bearer {{token}}"}],
        "url": {"raw": "{{base_url}}/libros", "host": ["{{base_url}}"], "path": ["libros"]}
      }
    },
    {
      "name": "4. Obtener libro",
      "event": [{
        "listen": "test",
        "script": {
          "type": "text/javascript",
          "exec": [
            "pm.test('Libro encontrado (200)', function () {",
            "  pm.response.to.have.status(200);",
            "});",
            "pm.test('Tiene el título esperado', function () {",
            "  pm.expect(pm.response.json().titulo).to.eql('Cien años de soledad');",
            "});"
          ]
        }
      }],
      "request": {
        "method": "GET",
        "header": [{"key": "Authorization", "value": "Bearer {{token}}"}],
        "url": {"raw": "{{base_url}}/libros/{{libro_id}}", "host": ["{{base_url}}"], "path": ["libros", "{{libro_id}}"]}
      }
    },
    {
      "name": "5. Actualizar libro",
      "event": [{
        "listen": "test",
        "script": {
          "type": "text/javascript",
          "exec": [
            "pm.test('Libro actualizado (200)', function () {",
            "  pm.response.to.have.status(200);",
            "});"
          ]
        }
      }],
      "request": {
        "method": "PUT",
        "header": [
          {"key": "Content-Type", "value": "application/json"},
          {"key": "Authorization", "value": "Bearer {{token}}"}
        ],
        "body": {
          "mode": "raw",
          "raw": "{\"titulo\": \"Cien años de soledad (ed. aniversario)\", \"autor\": \"Gabriel García Márquez\"}"
        },
        "url": {"raw": "{{base_url}}/libros/{{libro_id}}", "host": ["{{base_url}}"], "path": ["libros", "{{libro_id}}"]}
      }
    },
    {
      "name": "6. Eliminar libro",
      "event": [{
        "listen": "test",
        "script": {
          "type": "text/javascript",
          "exec": [
            "pm.test('Libro eliminado (204)', function () {",
            "  pm.response.to.have.status(204);",
            "});",
            "pm.environment.set('libro_id', '');"
          ]
        }
      }],
      "request": {
        "method": "DELETE",
        "header": [{"key": "Authorization", "value": "Bearer {{token}}"}],
        "url": {"raw": "{{base_url}}/libros/{{libro_id}}", "host": ["{{base_url}}"], "path": ["libros", "{{libro_id}}"]}
      }
    }
  ]
}


Writing coleccion.json


### 📄 `entorno.json`

Variables de entorno.

In [3]:
%%writefile entorno.json
{
  "name": "local",
  "values": [
    {"key": "base_url", "value": "http://127.0.0.1:8000"},
    {"key": "token", "value": ""},
    {"key": "libro_id", "value": ""}
  ]
}


Writing entorno.json


## 🧪 Tu contrato C-R-E-O

Antes de pedir ayuda a la IA, escribe el contrato que describe el resultado. Esto alimenta tu bitácora y te obliga a pensar como arquitecto:

| Elemento | Tu definición |
| :--- | :--- |
| **Contexto** | Framework, versión y restricciones del entorno. |
| **Rol** | Qué papel técnico debe asumir el modelo. |
| **Especificación** | Rutas, entradas, salidas, validaciones y códigos HTTP. |
| **Output** | Qué formato exacto esperas de vuelta. |

> No pegues código de la IA sin auditarlo. La responsabilidad técnica del resultado es tuya.


## 🔍 Verificación del resultado

Ejecuta esta celda cuando tu implementación esté lista. La verificación no modifica tu código: solo observa el comportamiento y compara con el contrato. Si algo falla, el mensaje te dice **qué esperaba** y **qué recibió**.


In [4]:
from verificar import revisar_json, resumen

resultados = []
resultados.append(revisar_json("coleccion.json", claves=["info.schema", "item"], contiene=["pm.response.to.have.status", "pm.environment.set", "{{token}}"], descripcion="La colección tiene aserciones y variables"))
resultados.append(revisar_json("entorno.json", claves=["values"], contiene=["base_url"], descripcion="El entorno define la URL base"))
resumen(resultados)


✅ La colección tiene aserciones y variables
✅ El entorno define la URL base
------------------------------------------------------------------------------
RESULTADO: 2/2 comprobaciones correctas. ¡Resultado logrado!


In [5]:
from verificar import revisar_python, resumen

resultados = []
resultados.append(revisar_python("import json\ndatos = json.load(open(\"coleccion.json\", encoding=\"utf-8\"))\nassert len(datos.get(\"item\", [])) >= 5, len(datos.get(\"item\", []))\n", "La colección tiene al menos cinco peticiones"))
resumen(resultados)


✅ La colección tiene al menos cinco peticiones
------------------------------------------------------------------------------
RESULTADO: 1/1 comprobaciones correctas. ¡Resultado logrado!


## 📝 Cierre y bitácora

1. Copia la plantilla `plantillas/auditoria_sesion37.md` a `auditorias/auditoria_sesion37.md` en tu fork.
2. Responde con tus palabras:
   - ¿Qué resultado te costó más alcanzar y por qué?
   - ¿Qué decisión de diseño tomaste que no estaba escrita en el contrato?
   - ¿Qué error detectó la verificación que tú no habías previsto?
3. Haz commit de tu cuaderno ejecutado y de tu bitácora.

**La sesión termina cuando el resultado está en verde, no cuando el código "parece funcionar".**
